<a href="https://colab.research.google.com/drive/1pDzpkkzqSYf9zfEj2AlElc3HmtaseVAv?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Paso 1: Instalar librerías de entrenamiento rápido
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.6/289.6 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 22.7 MB/s eta 0:00:00


In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# Configuración del modelo base
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# Cargar tu dataset recién subido
dataset = load_dataset("json", data_files="/content/dataset_tutor.jsonl", split="train")

print("¡Listo! Modelo y Dataset cargados. ¿Quieres que procedamos al entrenamiento?")

==((====))==  Unsloth 2025.12.7: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Generating train split: 0 examples [00:00, ? examples/s]

¡Listo! Modelo y Dataset cargados. ¿Quieres que procedamos al entrenamiento?


In [ ]:
# Añadimos los adaptadores LoRA para que el modelo pueda ser entrenado
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Elige cualquier número como 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Dropout 0 es óptimo para velocidad
    bias = "none",    # "none" es óptimo para velocidad
    use_gradient_checkpointing = "unsloth", # Reduce uso de memoria
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print(" Adaptadores añadidos. El modelo ahora está listo para entrenar.")

 Adaptadores añadidos. El modelo ahora está listo para entrenar.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Configuramos el entrenador (Trainer)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "instruction",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,            # Mantenemos 60 para evitar que memorice de más
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.1,        # <--- CORREGIDO: Evita la memorización exacta
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# ¡Iniciamos el aprendizaje del Tutor!
print("Iniciando el entrenamiento...")
trainer_stats = trainer.train()

Map (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 5 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Iniciando el entrenamiento...


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


wandb: WARNING Failed to wrap stdout. Console logs will not be captured.
wandb: WARNING Failed to wrap stderr. Console logs will not be captured.
wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
1,4.504100
2,4.133800
3,4.216900
4,4.010200
5,3.983700
6,4.370300
7,3.890500
8,3.817600
9,3.523100
10,3.384500


In [ ]:
def preguntar_tutor(pregunta):
    # Mantenemos tu estructura de Prompt pero con una instrucción más clara
    prompt = (
        "### Instruction:\n"
        "Eres un tutor de programación experto. Define el concepto en español y da un ejemplo de código Python válido.\n"
        f"Pregunta: {pregunta}\n\n"
        "### Response:\n"
    )

    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    # Ajuste de parámetros para evitar errores lógicos (Deteriminismo vs Creatividad)
    outputs = model.generate(
        **inputs,
        max_new_tokens = 120,           # Un poco más de espacio para el código
        use_cache = True,
        do_sample = True,
        temperature = 0.1,              # El balance ideal para que no invente mentiras
        top_p = 0.9,
        repetition_penalty = 1.1,       # Bajamos un poco para que el texto fluya mejor
        eos_token_id = tokenizer.eos_token_id
    )

    respuesta_raw = tokenizer.batch_decode(outputs)[0]

    if "### Response:" in respuesta_raw:
        respuesta = respuesta_raw.split("### Response:")[1].strip()

        # Tu lógica de limpieza original mejorada
        for marcador in ["Pregunta:", "###", "PREGUNTA:", "<|"]:
            if marcador in respuesta:
                respuesta = respuesta.split(marcador)[0].strip()

        respuesta = respuesta.replace("</s>", "").replace("<|end_of_text|>", "").strip()
        return respuesta

    return "No se pudo generar respuesta."

# --- PRUEBA CON EJEMPLO ---
duda = "¿Qué es un Booleano?"
print(f"RESPUESTA:\n{preguntar_tutor(duda)}")

RESPUESTA:
Un booleano es una variable que puede tener dos valores, True o False.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
